# Module 3 — LLM Generation & RAG / No-RAG

**Project:** RAG Evaluation: Effect of Chunking Strategy and Retrieval Parameters on Answer Quality

## What this notebook does
1. Loads Module 2's `retrieval_results` CSV.
2. Loads a Qwen instruction model.
3. Generates one answer with retrieved context (RAG).
4. Generates a baseline answer without retrieved context (No-RAG).
5. Saves a CSV that Module 4 can score.

**Input expected from Module 2:** `config_id`, `query`, `chunk_size`, `overlap`, `top_k`, `rank`, `paper_id`, `section`, `text`, and `similarity_score`.

Run this notebook in Google Colab with **Runtime → Change runtime type → T4 GPU** if available.

## Cell 1 — Install required packages



In [1]:

%pip install -q transformers accelerate pandas==2.2.3




## Cell 2 — Check GPU and import libraries

In [2]:
import os
import gc
import json
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. The model may run slowly on CPU.")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


## Cell 3 — Upload Module 2's CSV output

In Module 2 notebook, export the DataFrame with:

```python
retrieval_results.to_csv("module_2_retrieval_results.csv", index=False)
```


In [3]:
import zipfile
import os
from google.colab import files

uploaded = files.upload()


# Search for CSV files among uploaded and extracted files
csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
if not csv_files:
    raise ValueError("No CSV file found. Please upload the Module 2 retrieval-results CSV or a ZIP containing it.")

INPUT_CSV = csv_files[0]
retrieval_results = pd.read_csv(INPUT_CSV)
print("Loaded:", INPUT_CSV)
print("Shape:", retrieval_results.shape)
print("Columns:", retrieval_results.columns.tolist())
display(retrieval_results.head())

Saving module_2_retrieval_results.csv to module_2_retrieval_results.csv
Loaded: module_2_retrieval_results.csv
Shape: (11400, 13)
Columns: ['config_id', 'query_id', 'query', 'chunk_size', 'overlap', 'top_k', 'rank', 'window_id', 'parent_chunk_id', 'paper_id', 'section', 'text', 'similarity_score']


,config_id,query_id,query,chunk_size,overlap,top_k,rank,window_id,parent_chunk_id,paper_id,section,text,similarity_score
0,CFG_001,Q001,What problem does Retrieval-Augmented Generati...,250,0,1,1,C000594_W001,C000594,P10,1 Introduction,retrieval - augmented generation ( rag ) syste...,0.879269
1,CFG_001,Q002,What are the two main components combined in t...,250,0,1,1,C000114_W001,C000114,P03,I Introduction,how these components intricately collaborate t...,0.751279
2,CFG_001,Q003,What task does Dense Passage Retrieval (DPR) t...,250,0,1,1,C000077_W001,C000077,P02,5 Experiments: Passage Retrieval,"in this section, we evaluate the retrieval per...",0.816723
3,CFG_001,Q004,What is the main purpose of RAGAS?,250,0,1,1,C000166_W001,C000166,P03,VI-A Downstream Task,the core task of rag remains question answerin...,0.675176
4,CFG_001,Q005,What issue is investigated in 'Lost in the Mid...,250,0,1,1,C000153_W001,C000153,P03,IV-A Context Curation,redundant information can interfere with the f...,0.803355


## Cell 4 — Validate and prepare Module 2 data

This handles the long-format output from Module 2: each row is one retrieved chunk. Rows are grouped by question and experiment configuration to create the context for Qwen.

In [4]:
required_columns = {
    "query", "chunk_size", "overlap", "top_k", "rank",
    "paper_id", "section", "text", "similarity_score"
}
missing = required_columns - set(retrieval_results.columns)
if missing:
    raise ValueError(
        f"Missing expected Module 2 columns: {sorted(missing)}. "
        "Check that you uploaded retrieval_results, not the original documents CSV."
    )

# Some versions may not contain config_id; create a stable configuration label if absent.
if "config_id" not in retrieval_results.columns:
    retrieval_results["config_id"] = (
        "CH" + retrieval_results["chunk_size"].astype(str)
        + "_OV" + retrieval_results["overlap"].astype(str)
        + "_K" + retrieval_results["top_k"].astype(str)
    )

retrieval_results["rank"] = pd.to_numeric(retrieval_results["rank"], errors="coerce")
retrieval_results["top_k"] = pd.to_numeric(retrieval_results["top_k"], errors="coerce")
retrieval_results["chunk_size"] = pd.to_numeric(retrieval_results["chunk_size"], errors="coerce")
retrieval_results["overlap"] = pd.to_numeric(retrieval_results["overlap"], errors="coerce")
retrieval_results["similarity_score"] = pd.to_numeric(
    retrieval_results["similarity_score"], errors="coerce"
)

retrieval_results["text"] = retrieval_results["text"].fillna("").astype(str)
retrieval_results["query"] = retrieval_results["query"].fillna("").astype(str)

retrieval_results = retrieval_results[
    retrieval_results["query"].str.strip().ne("")
    & retrieval_results["text"].str.strip().ne("")
].copy()

print("Valid retrieval rows:", len(retrieval_results))
print("Unique questions:", retrieval_results["query"].nunique())
print("Configurations:", retrieval_results["config_id"].nunique())
print("Chunk sizes:", sorted(retrieval_results["chunk_size"].dropna().unique().tolist()))
print("Overlaps:", sorted(retrieval_results["overlap"].dropna().unique().tolist()))
print("Top-K values:", sorted(retrieval_results["top_k"].dropna().unique().tolist()))

Valid retrieval rows: 11400
Unique questions: 30
Configurations: 80
Chunk sizes: [250, 500, 750, 1000, 1500]
Overlaps: [0, 50, 100, 200]
Top-K values: [1, 3, 5, 10]


## Cell 5 — Load Qwen

In [5]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model_kwargs = {"device_map": "auto"}
if torch.cuda.is_available():
    model_kwargs["torch_dtype"] = torch.float16

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    **model_kwargs
)
model.eval()

print("Loaded model:", MODEL_NAME)
print("Input device:", model.get_input_embeddings().weight.device)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded model: Qwen/Qwen2.5-3B-Instruct
Input device: cuda:0


## Cell 6 — Define answer-generation functions

The RAG and No-RAG calls use the same model and generation settings. The difference is whether the prompt includes retrieved context.

The RAG prompt instructs the model to rely on the supplied excerpts and to say when the excerpts do not contain enough information. This instruction reduces, but cannot eliminate, hallucinations.

In [6]:
MAX_NEW_TOKENS = 220
TEMPERATURE = 0.0  # deterministic / greedy-style generation
MAX_INPUT_TOKENS = 6000  # keep context bounded for this model/runtime

def _generate_from_messages(messages, max_new_tokens=MAX_NEW_TOKENS):
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS
    )
    input_device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(input_device) for k, v in inputs.items()}

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return answer

def generate_no_rag_answer(query):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a careful research assistant. Answer the question clearly. "
                "Do not invent citations or claim to have read documents you were not given. "
                "If you are unsure, say so."
            )
        },
        {"role": "user", "content": query}
    ]
    return _generate_from_messages(messages)

def generate_rag_answer(query, retrieved_context):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a careful research-paper question-answering assistant. "
                "Answer using the supplied context. Do not add unsupported claims. "
                "If the context does not contain enough evidence, explicitly say that "
                "the provided excerpts are insufficient. Keep the answer concise."
            )
        },
        {
            "role": "user",
            "content": (
                f"Question:\n{query}\n\n"
                f"Retrieved research-paper excerpts:\n{retrieved_context}\n\n"
                "Answer the question using the excerpts above. If the answer is not supported "
                "by the excerpts, state that clearly."
            )
        }
    ]
    return _generate_from_messages(messages)

print("Generation functions are ready.")

Generation functions are ready.


## Cell 7 — Build context from ranked chunks

In [7]:
def build_context(group):
    group = group.sort_values("rank", ascending=True)
    parts = []
    for _, row in group.iterrows():
        paper_id = str(row.get("paper_id", "unknown"))
        section = str(row.get("section", "unknown"))
        rank = row.get("rank", "")
        text = str(row.get("text", "")).strip()
        if text:
            parts.append(
                f"[Retrieved chunk rank={rank}; paper_id={paper_id}; section={section}]\n{text}"
            )
    return "\n\n".join(parts)

# Module 2 has one row per retrieved chunk. This creates one group per question/configuration.
group_columns = ["config_id", "query", "chunk_size", "overlap", "top_k"]
experiment_groups = list(
    retrieval_results.groupby(group_columns, dropna=False, sort=False)
)

print("Question/configuration groups to process:", len(experiment_groups))
print("Example group keys:", experiment_groups[0][0] if experiment_groups else "No groups found")

Question/configuration groups to process: 2400
Example group keys: ('CFG_001', 'What problem does Retrieval-Augmented Generation (RAG) aim to address in knowledge-intensive NLP tasks?', np.int64(250), np.int64(0), np.int64(1))


## Cell 8 — Run a small pilot first

In [8]:
PILOT_GROUPS = min(3, len(experiment_groups))
no_rag_cache = {}
pilot_rows = []

for group_index, (keys, group) in enumerate(experiment_groups[:PILOT_GROUPS], start=1):
    config_id, query, chunk_size, overlap, top_k = keys
    context = build_context(group)

    if query not in no_rag_cache:
        no_rag_cache[query] = generate_no_rag_answer(query)

    rag_answer = generate_rag_answer(query, context)

    pilot_rows.append({
        "config_id": config_id,
        "query": query,
        "chunk_size": chunk_size,
        "overlap": overlap,
        "top_k": top_k,
        "rag_answer": rag_answer,
        "no_rag_answer": no_rag_cache[query],
        "retrieved_context": context
    })
    print(f"Completed pilot {group_index}/{PILOT_GROUPS}")

pilot_df = pd.DataFrame(pilot_rows)
display(pilot_df[["query", "chunk_size", "overlap", "top_k", "rag_answer", "no_rag_answer"]])

Completed pilot 1/3
Completed pilot 2/3
Completed pilot 3/3


,query,chunk_size,overlap,top_k,rag_answer,no_rag_answer
0,What problem does Retrieval-Augmented Generati...,250,0,1,Retrieval-Augmented Generation (RAG) aims to a...,Retrieval-Augmented Generation (RAG) aims to a...
1,What are the two main components combined in t...,250,0,1,The two main components combined in the RAG (R...,"The RAG (Red, Amber, Green) approach is common..."
2,What task does Dense Passage Retrieval (DPR) t...,250,0,1,Dense Passage Retrieval (DPR) targets the task...,Dense Passage Retrieval (DPR) targets the task...


## Cell 9 — Run the full generation experiment

In [9]:
all_result_rows = []
no_rag_cache = {}

for group_index, (keys, group) in enumerate(experiment_groups, start=1):
    config_id, query, chunk_size, overlap, top_k = keys
    group = group.sort_values("rank", ascending=True)
    context = build_context(group)

    try:
        if query not in no_rag_cache:
            no_rag_cache[query] = generate_no_rag_answer(query)
        no_rag_answer = no_rag_cache[query]
        rag_answer = generate_rag_answer(query, context)
        status = "success"
        error_message = ""
    except Exception as e:
        rag_answer = ""
        no_rag_answer = no_rag_cache.get(query, "")
        status = "error"
        error_message = repr(e)
        print(f"Error on group {group_index} ({config_id}): {error_message}")

    paper_ids = [
        str(x) for x in group["paper_id"].dropna().drop_duplicates().tolist()
    ]
    chunk_ids = (
        group["parent_chunk_id"].dropna().astype(str).tolist()
        if "parent_chunk_id" in group.columns
        else []
    )
    window_ids = (
        group["window_id"].dropna().astype(str).tolist()
        if "window_id" in group.columns
        else []
    )
    similarity_scores = group["similarity_score"].dropna().tolist()

    all_result_rows.append({
        "config_id": config_id,
        "query": query,
        "chunk_size": chunk_size,
        "overlap": overlap,
        "top_k": top_k,
        "num_retrieved_chunks": len(group),
        "retrieved_paper_ids": json.dumps(paper_ids, ensure_ascii=False),
        "retrieved_parent_chunk_ids": json.dumps(chunk_ids, ensure_ascii=False),
        "retrieved_window_ids": json.dumps(window_ids, ensure_ascii=False),
        "similarity_scores": json.dumps(similarity_scores),
        "retrieved_context": context,
        "rag_answer": rag_answer,
        "no_rag_answer": no_rag_answer,
        "model_name": MODEL_NAME,
        "max_new_tokens": MAX_NEW_TOKENS,
        "generation_mode": "greedy",
        "generation_status": status,
        "error_message": error_message
    })

    if group_index % 5 == 0 or group_index == len(experiment_groups):
        print(f"Processed {group_index}/{len(experiment_groups)} groups")

generation_results = pd.DataFrame(all_result_rows)
print("Generation result shape:", generation_results.shape)
display(generation_results.head())

Processed 5/2400 groups
Processed 10/2400 groups
Processed 15/2400 groups
Processed 20/2400 groups
Processed 25/2400 groups
Processed 30/2400 groups
Processed 35/2400 groups
Processed 40/2400 groups
Processed 45/2400 groups
Processed 50/2400 groups
Processed 55/2400 groups
Processed 60/2400 groups
Processed 65/2400 groups
Processed 70/2400 groups
Processed 75/2400 groups
Processed 80/2400 groups
Processed 85/2400 groups
Processed 90/2400 groups
Processed 95/2400 groups
Processed 100/2400 groups
Processed 105/2400 groups
Processed 110/2400 groups
Processed 115/2400 groups
Processed 120/2400 groups
Processed 125/2400 groups
Processed 130/2400 groups
Processed 135/2400 groups
Processed 140/2400 groups
Processed 145/2400 groups
Processed 150/2400 groups
Processed 155/2400 groups
Processed 160/2400 groups
Processed 165/2400 groups
Processed 170/2400 groups
Processed 175/2400 groups
Processed 180/2400 groups
Processed 185/2400 groups
Processed 190/2400 groups
Processed 195/2400 groups
Proces

,config_id,query,chunk_size,overlap,top_k,num_retrieved_chunks,retrieved_paper_ids,retrieved_parent_chunk_ids,retrieved_window_ids,similarity_scores,retrieved_context,rag_answer,no_rag_answer,model_name,max_new_tokens,generation_mode,generation_status,error_message
0,CFG_001,What problem does Retrieval-Augmented Generati...,250,0,1,1,"[""P10""]","[""C000594""]","[""C000594_W001""]",[0.879269003868103],[Retrieved chunk rank=1; paper_id=P10; section...,Retrieval-Augmented Generation (RAG) aims to a...,Retrieval-Augmented Generation (RAG) aims to a...,Qwen/Qwen2.5-3B-Instruct,220,greedy,success,
1,CFG_001,What are the two main components combined in t...,250,0,1,1,"[""P03""]","[""C000114""]","[""C000114_W001""]",[0.7512792348861694],[Retrieved chunk rank=1; paper_id=P03; section...,The two main components combined in the RAG (R...,"The RAG (Red, Amber, Green) approach is common...",Qwen/Qwen2.5-3B-Instruct,220,greedy,success,
2,CFG_001,What task does Dense Passage Retrieval (DPR) t...,250,0,1,1,"[""P02""]","[""C000077""]","[""C000077_W001""]",[0.8167228698730469],[Retrieved chunk rank=1; paper_id=P02; section...,Dense Passage Retrieval (DPR) targets the task...,Dense Passage Retrieval (DPR) targets the task...,Qwen/Qwen2.5-3B-Instruct,220,greedy,success,
3,CFG_001,What is the main purpose of RAGAS?,250,0,1,1,"[""P03""]","[""C000166""]","[""C000166_W001""]",[0.6751759052276611],[Retrieved chunk rank=1; paper_id=P03; section...,"The main purpose of RAGAS, as per the provided...",I do not have specific information about an or...,Qwen/Qwen2.5-3B-Instruct,220,greedy,success,
4,CFG_001,What issue is investigated in 'Lost in the Mid...,250,0,1,1,"[""P03""]","[""C000153""]","[""C000153_W001""]",[0.8033552765846252],[Retrieved chunk rank=1; paper_id=P03; section...,The issue investigated in 'Lost in the Middle:...,"The title ""Lost in the Middle: How Language Mo...",Qwen/Qwen2.5-3B-Instruct,220,greedy,success,


## Cell 10 — Quality checks

In [10]:
if not generation_results.empty:
    print("Rows:", len(generation_results))
    print("Successful:", int((generation_results["generation_status"] == "success").sum()))
    print("Errors:", int((generation_results["generation_status"] == "error").sum()))
    print("Missing RAG answers:", int(generation_results["rag_answer"].fillna("").str.strip().eq("").sum()))
    print("Missing No-RAG answers:", int(generation_results["no_rag_answer"].fillna("").str.strip().eq("").sum()))
    display(
        generation_results[
            ["query", "chunk_size", "overlap", "top_k", "generation_status"]
        ].head(10)
    )
else:
    print("No generated results. Check the uploaded CSV and rerun the generation cells.")

Rows: 2400
Successful: 2400
Errors: 0
Missing RAG answers: 0
Missing No-RAG answers: 0


,query,chunk_size,overlap,top_k,generation_status
0,What problem does Retrieval-Augmented Generati...,250,0,1,success
1,What are the two main components combined in t...,250,0,1,success
2,What task does Dense Passage Retrieval (DPR) t...,250,0,1,success
3,What is the main purpose of RAGAS?,250,0,1,success
4,What issue is investigated in 'Lost in the Mid...,250,0,1,success
5,What architecture is introduced in 'Attention ...,250,0,1,success
6,"What does BERT stand for, and what kind of rep...",250,0,1,success
7,What learning setting is demonstrated by GPT-3...,250,0,1,success
8,What does LoRA change when adapting a large la...,250,0,1,success
9,What is the goal of RAGChecker?,250,0,1,success


## Cell 11 — Export Module 3 results for Module 4

In [11]:
OUTPUT_CSV = "module_3_generation_results.csv"
generation_results.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print("Saved:", OUTPUT_CSV)

from google.colab import files
files.download(OUTPUT_CSV)

Saved: module_3_generation_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>